In [1]:
# Task 7 & 8: Strict Compliance Virtual Execution (May 2026)

import yfinance as yf
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# 1. Load Portfolio Allocations
print("Loading Task 5 portfolio allocations...")
allocation = pd.read_csv('../reports/task5_final_allocation.csv')
portfolio_tickers = allocation['Stock'].tolist()

# 2. Fetch STRICT Training Data (Ends Dec 31, 2025) and Actual May Data
print("Fetching strict historical data and actual May market data...")
train_data = yf.download(portfolio_tickers, start="2021-01-01", end="2025-12-31")['Close']
train_data = train_data.ffill().bfill()

actual_data = yf.download(portfolio_tickers, start="2026-05-14", end="2026-05-16")['Close']
actual_data = actual_data.ffill().bfill()

day1_date = actual_data.index[0] # May 14
day2_date = actual_data.index[1] # May 15

# Calculate how many business days are between Dec 31, 2025 and May 15, 2026
# This tells the model exactly how many steps into the future to predict
forecast_horizon = len(pd.bdate_range(start='2026-01-01', end='2026-05-15'))

print(f"\nForecasting {forecast_horizon} days into the future based strictly on 2025 data...\n")
comparison_results = []

for index, row in allocation.iterrows():
    stock = row['Stock']
    capital = row['Allocated Amount (₹)']
    
    # --- A. Train on strictly 2025 data and Long-Range Forecast ---
    series = train_data[stock].dropna()
    try:
        exp_model = ExponentialSmoothing(series, trend='add', initialization_method="estimated")
        exp_fit = exp_model.fit()
        
        # Forecast the massive gap
        long_forecast = exp_fit.forecast(forecast_horizon)
        
        # Grab the very last two days (May 14 and May 15)
        pred_day1 = long_forecast.iloc[-2]
        pred_day2 = long_forecast.iloc[-1]
    except Exception as e:
        pred_day1, pred_day2 = np.nan, np.nan
        
    # --- B. Get Actuals ---
    actual_day1 = actual_data.loc[day1_date, stock]
    actual_day2 = actual_data.loc[day2_date, stock]
    
    # --- C. Calculate P&L (Task 7 & 8) ---
    shares_bought = capital / actual_day1
    final_value = shares_bought * actual_day2
    profit_loss = final_value - capital
    actual_return_pct = ((actual_day2 - actual_day1) / actual_day1) * 100
    
    comparison_results.append({
        'Stock': stock,
        'Pred Day 1': round(pred_day1, 2),
        'Actual Day 1': round(actual_day1, 2),
        'Pred Day 2': round(pred_day2, 2),
        'Actual Day 2': round(actual_day2, 2),
        'Actual Return (%)': round(actual_return_pct, 2),
        'P&L (₹)': round(profit_loss, 2)
    })

comparison_df = pd.DataFrame(comparison_results)

# 4. Final Portfolio Math
total_initial = allocation['Allocated Amount (₹)'].sum()
total_profit = comparison_df['P&L (₹)'].sum()
total_final = total_initial + total_profit
total_return_pct = (total_profit / total_initial) * 100

comparison_df.to_csv('../reports/task8_strict_compliance_performance.csv', index=False)

# --- Display Final Results ---
print("=======================================================")
print("TASK 8: PREDICTED VS ACTUAL OUTCOMES (MAY 14-15)")
print("=======================================================\n")
print(comparison_df[['Stock', 'Pred Day 1', 'Actual Day 1', 'Pred Day 2', 'Actual Day 2', 'P&L (₹)']].to_string(index=False))

print("\n=======================================================")
print("FINAL PORTFOLIO PERFORMANCE SUMMARY")
print("=======================================================")
print(f"Initial Capital Deployed : ₹{total_initial:,.2f}")
print(f"Final Portfolio Value    : ₹{total_final:,.2f}")
print(f"Total Profit / Loss      : ₹{total_profit:,.2f}")
print(f"Net Portfolio Return     : {total_return_pct:.2f}%")
print("=======================================================\n")

Loading Task 5 portfolio allocations...
Fetching strict historical data and actual May market data...


[*********************100%***********************]  26 of 26 completed
[*********************100%***********************]  26 of 26 completed



Forecasting 97 days into the future based strictly on 2025 data...

TASK 8: PREDICTED VS ACTUAL OUTCOMES (MAY 14-15)

        Stock  Pred Day 1  Actual Day 1  Pred Day 2  Actual Day 2  P&L (₹)
      OFSS.NS     7800.26       8904.50     7804.28       9015.00  1352.20
   J&KBANK.NS      105.07        132.05      105.13        130.70 -1104.66
   MANKIND.NS     2210.15       2462.20     2210.75       2503.00  1122.58
     SPARC.NS      129.62        170.08      129.56        163.65 -2470.66
ADANIPORTS.NS     1537.14       1773.40     1537.93       1795.10   677.97
  ASIANENE.NS      288.37        306.25      288.48        313.45  1263.85
        LT.NS     4271.41       3940.40     4273.70       3909.00  -330.99
   CARYSIL.NS      940.55        912.35      941.11        918.45   271.96
   SBILIFE.NS     2078.77       1866.70     2079.66       1864.50   -46.69
      SBIN.NS     1010.36        962.55     1010.93        963.20    23.81
BAJAJ-AUTO.NS     9768.79      10451.00     9773.86     